## MlLib library check

### Importing the necessary libraries

In [1]:
from pyspark.sql import SparkSession

from pyspark.ml.feature import VectorAssembler
from pyspark.ml.classification import LogisticRegression


In [2]:
# Creating Spark session
spark = SparkSession.builder.appName("MLlib_Test").getOrCreate()
spark


In [3]:

# Sample dataset
data = [
    (0, 1.0, 0.1, 0),
    (1, 2.0, 1.1, 0),
    (2, 3.0, 2.1, 1),
    (3, 4.0, 3.1, 1)
]

columns = ["id", "feature1", "feature2", "label"]

df = spark.createDataFrame(data, columns)

print("Original Data:")
df.show()


Original Data:
+---+--------+--------+-----+
| id|feature1|feature2|label|
+---+--------+--------+-----+
|  0|     1.0|     0.1|    0|
|  1|     2.0|     1.1|    0|
|  2|     3.0|     2.1|    1|
|  3|     4.0|     3.1|    1|
+---+--------+--------+-----+



In [4]:
# Combine features into vector
assembler = VectorAssembler(
    inputCols=["feature1", "feature2"],
    outputCol="features"
)

df_vector = assembler.transform(df)

print("Vectorized Data:")
df_vector.select("features", "label").show(truncate=False)

# Train Logistic Regression model
lr = LogisticRegression(
    featuresCol="features",
    labelCol="label"
)

model = lr.fit(df_vector)

print("Model trained successfully!")

# Make predictions
predictions = model.transform(df_vector)

print("Predictions:")
predictions.select(
    "features",
    "label",
    "prediction",
    "probability"
).show(truncate=False)

Vectorized Data:
+---------+-----+
|features |label|
+---------+-----+
|[1.0,0.1]|0    |
|[2.0,1.1]|0    |
|[3.0,2.1]|1    |
|[4.0,3.1]|1    |
+---------+-----+

Model trained successfully!
Predictions:
+---------+-----+----------+------------------------------------------+
|features |label|prediction|probability                               |
+---------+-----+----------+------------------------------------------+
|[1.0,0.1]|0    |0.0       |[1.0,0.0]                                 |
|[2.0,1.1]|0    |0.0       |[0.999999967244475,3.2755524959071636E-8] |
|[3.0,2.1]|1    |1.0       |[3.2704284774795586E-8,0.9999999672957153]|
|[4.0,3.1]|1    |1.0       |[3.5034338173605935E-23,1.0]              |
+---------+-----+----------+------------------------------------------+

